# Convolutional Autoencoder for Image Denoising
## Neural networks for energy applications — Day 4, Exercise 3

In this exercise we build a **convolutional autoencoder** that learns to denoise
images. We train the network to reconstruct clean MNIST digits from noisy inputs,
using Conv2d layers in the encoder and ConvTranspose2d layers in the decoder.
We then explore the latent space to understand what the model has learned.

**Evaluation protocol.** We hold out 5,000 of the 60,000 training images as a
**validation set**. Validation images are used to monitor training, to keep the best
epoch and to compare latent sizes, always with the same fixed noise. The MNIST
**test set** is used once, in Section 8, for the model we have already chosen.

**Runtime.** Budgets are set for a laptop CPU: 8 epochs for the main model and
3 epochs for each model in the latent-size comparison.
The notebook runs in about 5–8 minutes on a laptop CPU and in about 1 minute on a GPU
or Apple silicon. With more time, raise `NUM_EPOCHS` and `COMPARE_EPOCHS`.

---
## 0 — Imports and Setup

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms

from sklearn.manifold import TSNE

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Use a GPU if there is one (CUDA, or MPS on Apple silicon), otherwise the CPU.
# GPU results can differ slightly from CPU results even with the same seed.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# MNIST is downloaded to ./data unless the COURSE_DATA environment variable names another folder
DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
print(f"Data folder: {DATA_ROOT}")
print("Setup complete.")

---
## 1 — Load MNIST Dataset

In [ ]:
# ToTensor scales pixels to [0, 1]; there is no mean/std normalization, so a Sigmoid output fits the target
transform = transforms.Compose([
    transforms.ToTensor(),
])

full_train_dataset = torchvision.datasets.MNIST(
    root=DATA_ROOT, train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root=DATA_ROOT, train=False, download=True, transform=transform
)

# Hold out 5,000 training images for validation; the test set is kept for the final check (Section 8)
train_dataset, val_dataset = random_split(
    full_train_dataset, [55000, 5000], generator=torch.Generator().manual_seed(RANDOM_STATE)
)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)} (kept for the final check)")
print(f"Image shape        : {full_train_dataset[0][0].shape}")

---
## 2 — Add Noise to Images

We add Gaussian noise to the input images. The autoencoder will learn to map
noisy images back to clean originals.

Training batches get **fresh noise every time** they are drawn. The validation images
get **one fixed noise draw** (seeded), so that every epoch and every model in Section 6
is scored on exactly the same noisy inputs; otherwise part of a difference between two
models would just be different noise.

In [ ]:
NOISE_FACTOR = 0.5


def add_noise(images, noise_factor=NOISE_FACTOR, generator=None):
    """Add Gaussian noise to images and clamp to [0, 1].

    Without a generator the noise is new on every call (training batches).
    With a seeded torch.Generator the noise is reproducible (validation and test images).
    """
    if generator is None:
        noise = torch.randn_like(images)
    else:
        noise = torch.randn(images.shape, generator=generator).to(images.device)
    return torch.clamp(images + noise_factor * noise, 0.0, 1.0)


# Fixed noisy validation images: every model below is scored on exactly the same inputs
val_clean = torch.stack([image for image, _ in val_dataset])
val_labels = torch.tensor([label for _, label in val_dataset])
val_noisy = add_noise(val_clean, generator=torch.Generator().manual_seed(RANDOM_STATE))
val_loader = DataLoader(TensorDataset(val_noisy, val_clean), batch_size=BATCH_SIZE, shuffle=False)

# Visualize clean vs noisy validation images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(val_clean[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title(f"Clean ({val_labels[i].item()})")

    axes[1, i].imshow(val_noisy[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
    axes[1, i].set_title("Noisy")

plt.suptitle(f"Clean vs Noisy Images (noise_factor={NOISE_FACTOR})", fontsize=14)
plt.tight_layout()
plt.show()

# Reference: the error of doing nothing (returning the noisy input unchanged)
print(f"Validation MSE of the noisy inputs against the clean images: {torch.mean((val_noisy - val_clean) ** 2):.4f}")

---
## 3 — Build Convolutional Autoencoder

The **encoder** compresses the 28x28 image down to a latent representation using
Conv2d layers. The **decoder** reconstructs the image using ConvTranspose2d layers.

Architecture:

| Component | Layer | Output Shape |
|-----------|-------|--------------|
| Encoder | Conv2d(1, 16, 3, stride=2, pad=1) + ReLU | 16 x 14 x 14 |
| Encoder | Conv2d(16, 32, 3, stride=2, pad=1) + ReLU | 32 x 7 x 7 |
| Encoder | Conv2d(32, 64, 7) + ReLU | 64 x 1 x 1 |
| Decoder | ConvTranspose2d(64, 32, 7) + ReLU | 32 x 7 x 7 |
| Decoder | ConvTranspose2d(32, 16, 3, stride=2, pad=1, out_pad=1) + ReLU | 16 x 14 x 14 |
| Decoder | ConvTranspose2d(16, 1, 3, stride=2, pad=1, out_pad=1) + Sigmoid | 1 x 28 x 28 |

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super(ConvAutoencoder, self).__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # -> 16x14x14
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # -> 32x7x7
            nn.ReLU(),
            nn.Conv2d(32, latent_dim, kernel_size=7),              # -> latent_dim x1x1
            nn.ReLU(),
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 32, kernel_size=7),                            # -> 32x7x7
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> 16x14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),   # -> 1x28x28
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)

model = ConvAutoencoder(latent_dim=64).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Verify output shape
dummy = torch.randn(1, 1, 28, 28).to(device)
out = model(dummy)
print(f"Input shape  : {dummy.shape}")
print(f"Latent shape : {model.encode(dummy).shape}")
print(f"Output shape : {out.shape}")

---
## 4 — Train the Denoising Autoencoder

We train with **MSE loss** between the reconstructed output and the clean original
image. The input to the model is the noisy version.

After every epoch we compute the MSE on the fixed noisy **validation** images and keep a
copy of the weights from the epoch with the lowest validation MSE. The test set plays
no part in this choice.

### 4.1 Training Loop

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def denoising_mse(model, loader, criterion):
    """Mean MSE between the reconstructions of the noisy inputs and the clean targets."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for noisy, clean in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            total_loss += criterion(model(noisy), clean).item() * clean.size(0)
    return total_loss / len(loader.dataset)


NUM_EPOCHS = 8   # classroom budget; more epochs usually give sharper reconstructions
history = {"train_loss": [], "val_loss": []}
best_val_loss, best_epoch, best_state = float("inf"), 0, None

for epoch in range(1, NUM_EPOCHS + 1):
    # Training: fresh noise for every batch, the CLEAN image is the target
    model.train()
    train_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        noisy = add_noise(images)

        optimizer.zero_grad()
        outputs = model(noisy)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation: the same fixed noisy images every epoch
    val_loss = denoising_mse(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    if val_loss < best_val_loss:
        best_val_loss, best_epoch = val_loss, epoch
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:>2d}/{NUM_EPOCHS}  Train Loss: {train_loss:.6f}  Val Loss: {val_loss:.6f}")

model.load_state_dict(best_state)
print(f"\nKeeping the weights from epoch {best_epoch} (validation MSE {best_val_loss:.6f})")

### 4.2 Plot Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
epochs = range(1, NUM_EPOCHS + 1)
ax.plot(epochs, history["train_loss"], "o-", label="Train Loss", color="steelblue")
ax.plot(epochs, history["val_loss"], "s--", label="Validation Loss", color="tomato")
ax.axvline(best_epoch, color="gray", linestyle=":", label=f"Kept epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Denoising Autoencoder Training Curve")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5 — Visualize Denoising Results

We compare noisy input, reconstructed output, clean original and the absolute
**residual** |reconstruction − clean| for eight validation digits. Structure in the
residual (strokes, edges) means the model misses signal; noise-like residuals mean it
does not.

In [ ]:
model.eval()
n_show = 8
with torch.no_grad():
    reconstructed = model(val_noisy[:n_show].to(device)).cpu()
residual = (reconstructed - val_clean[:n_show]).abs()

rows = [
    ("Noisy", val_noisy),
    ("Reconstructed", reconstructed),
    ("Clean", val_clean),
    ("|Residual|", residual),
]
fig, axes = plt.subplots(len(rows), n_show, figsize=(16, 8))
for r, (name, images) in enumerate(rows):
    for i in range(n_show):
        axes[r, i].imshow(images[i].squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[r, i].set_xticks([])
        axes[r, i].set_yticks([])
    axes[r, 0].set_ylabel(name, fontsize=12)

plt.suptitle("Denoising Autoencoder: Noisy -> Reconstructed -> Clean, and Residual", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6 — Experiment with Latent Space Dimension

We train autoencoders with latent sizes 8, 32 and 64 to see how the compression level
affects reconstruction quality. For a fair comparison every size gets the **same
training budget** (3 epochs, the same seed) and is scored on the **same fixed noisy
validation images**.

In [ ]:
latent_dims = [8, 32, 64]
COMPARE_EPOCHS = 3   # the same, shorter budget for every latent size
results = {}

for ldim in latent_dims:
    print(f"\n--- Training with latent_dim={ldim} ---")
    torch.manual_seed(RANDOM_STATE)   # same initial seed and batch order for every size
    ae = ConvAutoencoder(latent_dim=ldim).to(device)
    opt = optim.Adam(ae.parameters(), lr=1e-3)

    for epoch in range(1, COMPARE_EPOCHS + 1):
        ae.train()
        for images, _ in train_loader:
            images = images.to(device)
            noisy = add_noise(images)
            opt.zero_grad()
            loss = criterion(ae(noisy), images)
            loss.backward()
            opt.step()

    val_mse = denoising_mse(ae, val_loader, criterion)
    n_params = sum(p.numel() for p in ae.parameters())
    results[ldim] = {"model": ae, "val_loss": val_mse, "params": n_params}
    print(f"latent_dim={ldim}  Validation MSE: {val_mse:.6f}")

print(f"\n--- Comparison (validation images, {COMPARE_EPOCHS} epochs each) ---")
for ldim, r in results.items():
    print(f"latent_dim={ldim:>4d}  parameters: {r['params']:>8,}  Validation MSE: {r['val_loss']:.6f}")

In [ ]:
# Visualize reconstructions of the same validation images for each latent dimension
n_show = 8
fig, axes = plt.subplots(len(latent_dims) + 2, n_show, figsize=(16, 2.5 * (len(latent_dims) + 2)))

noisy_batch = val_noisy[:n_show].to(device)

# Row 0: noisy input
for i in range(n_show):
    axes[0, i].imshow(val_noisy[i].squeeze(), cmap="gray")
axes[0, 0].set_ylabel("Noisy", fontsize=11)

# Rows 1..N: reconstructed for each latent dim
for row, ldim in enumerate(latent_dims, start=1):
    ae = results[ldim]["model"]
    ae.eval()
    with torch.no_grad():
        recon = ae(noisy_batch).cpu()
    for i in range(n_show):
        axes[row, i].imshow(recon[i].squeeze(), cmap="gray")
    axes[row, 0].set_ylabel(f"dim={ldim}", fontsize=11)

# Last row: clean original
for i in range(n_show):
    axes[-1, i].imshow(val_clean[i].squeeze(), cmap="gray")
axes[-1, 0].set_ylabel("Clean", fontsize=11)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Reconstruction Quality vs Latent Dimension (validation images)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 7 — Visualize Latent Representations

We extract the latent vectors of the validation images (encoded from the clean images)
and use t-SNE to project them into 2D, colouring by digit class.

t-SNE is a visualization aid only: the distances between clusters and their sizes are
not meaningful, and a tidy map is not evidence that the codes are useful for another task.

In [ ]:
# Extract latent representations of the validation images using the main model (latent_dim=64)
model.eval()
latent_vectors = []
all_labels = []

val_code_loader = DataLoader(TensorDataset(val_clean, val_labels), batch_size=BATCH_SIZE, shuffle=False)
with torch.no_grad():
    for images, labels in val_code_loader:
        images = images.to(device)
        z = model.encode(images)  # shape: (batch, 64, 1, 1)
        latent_vectors.append(z.flatten(start_dim=1).cpu().numpy())
        all_labels.append(labels.numpy())

latent_vectors = np.concatenate(latent_vectors, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

print(f"Latent vectors shape: {latent_vectors.shape}")
print(f"Labels shape        : {all_labels.shape}")

In [ ]:
# t-SNE on a subset (2000 samples for speed)
n_subset = 2000
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(latent_vectors), n_subset, replace=False)
latent_subset = latent_vectors[idx]
labels_subset = all_labels[idx]

tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
latent_2d = tsne.fit_transform(latent_subset)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    latent_2d[:, 0], latent_2d[:, 1],
    c=labels_subset, cmap="tab10", s=10, alpha=0.7
)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_label("Digit Class")
ax.set_xlabel("t-SNE Component 1")
ax.set_ylabel("t-SNE Component 2")
ax.set_title("t-SNE of Autoencoder Latent Space (2000 validation samples)")
plt.tight_layout()
plt.show()

---
## 8 — Final Check on the Test Set (once)

All choices are now locked: the latent-64 model from Section 4 with the weights of its
best validation epoch. (Had the comparison in Section 6 favoured another size, we would
retrain that size with the full budget and check it on validation data before this step.)

We now use the MNIST test set for the first and only time, with its own fixed noise, and
report the test MSE next to the "do nothing" reference: the MSE of the noisy input itself.

In [ ]:
test_clean = torch.stack([image for image, _ in test_dataset])
test_noisy = add_noise(test_clean, generator=torch.Generator().manual_seed(RANDOM_STATE + 1))
test_loader = DataLoader(TensorDataset(test_noisy, test_clean), batch_size=BATCH_SIZE, shuffle=False)

noisy_input_mse = torch.mean((test_noisy - test_clean) ** 2).item()
test_mse = denoising_mse(model, test_loader, criterion)

print(f"Test MSE, noisy input vs clean (do-nothing reference): {noisy_input_mse:.4f}")
print(f"Test MSE, reconstruction vs clean (final model)       : {test_mse:.4f}")

---
## Summary

In this exercise we:

1. **Loaded MNIST**, held out 5,000 training images for validation and added Gaussian noise.
2. **Built a convolutional autoencoder** with Conv2d encoder and ConvTranspose2d decoder.
3. **Trained** the model to reconstruct clean images from noisy inputs using MSE loss,
   monitoring fixed noisy validation images and keeping the best validation epoch.
4. **Visualized** denoising results: noisy input, reconstruction, clean original and residual.
5. **Compared** latent sizes 8, 32 and 64 with the same training budget on the same
   validation images.
6. **Visualized** the latent space of validation images using t-SNE.
7. **Checked the locked model once on the test set**, against the do-nothing reference.

Key takeaways:
- Convolutional autoencoders preserve spatial structure better than fully connected ones.
- The latent dimension controls the information bottleneck: too small loses detail,
  too large may not denoise as effectively.
- Compare models on the same validation images with the same noise and the same
  training budget; the test set is for one final check.
- Similar digits tend to lie close together in the t-SNE map, but t-SNE distances and
  cluster sizes are not evidence about the latent space on their own.